# Marginal Impact — 200 RP OEP, window method**Inputs**| object | server | grain ||---|---|---|| Portfolio ELT | RMS | one row per event || Account ELT | RMS | one row per event, per account || `VendorEventsMap` + `YEQT` | QBE | one `LossQuantile` per (SimID, EventID) |**Method** (manager's 8 steps, OEP basis)1. Simulate the **portfolio** ELT against YEQT → portfolio YELT (event level)2. Year OEP = max event loss per SimID → sort desc → 200 RP at rank `N/200`3. Window = **top-K years**, carrying *all* their events4. Simulate the **account** ELT restricted to window (year, event)5. Subtract on (SimID, EventID)6. Recompute per-year max — demotion-safe7. New 200 RP = rank `N/200` of the window8. Marginal = original − new**Two design points that are load-bearing**- The window is **ranks 1…K**, not a band around rank 2,500. Removal only lowers  losses, so the post-removal rank-2,500 year can rise from anywhere above it.- Window years carry **every** event, not just their argmax. Removing an account  can demote the largest event below another event in the same year.**Exactness certificate.** Let `M_{K+1}` be the largest original year OEP *outside*the window. Every outside year has post-removal OEP ≤ its original OEP ≤ `M_{K+1}`.So if the post-removal rank-2500 value ≥ `M_{K+1}`, no outside year can enter thetop 2500 and **the windowed answer is exact, not approximate**. Asserted per account.

## 1 · Setup

In [1]:
from __future__ import annotations
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# --- run configuration -----------------------------------------------------
N_SIMS        = 500_000     # simulation years in the YEQT
RETURN_PERIOD = 200
K_WINDOW      = 25_000      # top-K years; 5% of sims. Tune after first run.

ANLSID_PORTFOLIO = 112      # RMS analysis holding the portfolio ELT
ID_PORTFOLIO     = None     #   ... and its id  (set me)
PERSPCODE        = "RL"

VENDORVERID   = 33          # must match the RMS model version behind ANLSID
HAZARDZONEID  = 57
YEQTID        = 69

PARAM_FLOOR = 1e-7
MU_CAP      = 1.0 - 1e-7
TOL         = 1e-6

TARGET_RANK = int(round(N_SIMS / RETURN_PERIOD))
print(f"target rank = {TARGET_RANK:,}   window K = {K_WINDOW:,}")
assert K_WINDOW > TARGET_RANK, "window must extend past the target rank"

target rank = 2,500   window K = 25,000


## 2 · ExtractsThe only cross-server hop is `eventid → VENDOREVENTID`. Everything else staysinside its own server.**Do not merge the full ELT against the full mapping.** `staging_df` holds everyaccount under the analysis; merging it whole against `mapping_df` materialises`n_accounts × n_events × n_sims` rows — 139.8M in the first attempt, and that isbefore any account is windowed. Filter to one `id` first, and for account runsrestrict the mapping to the window years before merging. The merge is then`account_events × sims_per_event`, which is small.

In [2]:
# from sqlalchemy import create_engine
#
# RMS_CONN = ("mssql+pyodbc://@<server>/<rdm_database>"
#             "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes")
# QBE_CONN = ("mssql+pyodbc://@<server>/gp_reference_dev"
#             "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes")
#
# engine_rdm = create_engine(RMS_CONN)    # RMS server  (rdm_*)
# engine_ref = create_engine(QBE_CONN)    # QBE server  (gp_reference_dev)

SQL_ELT = """
SELECT
      ra.ANLSID, ra.ID, ra.EVENTID AS VENDOREVENTID, ra.PERSPCODE
    , ra.PERSPVALUE                      -- mean loss
    , ras.STDDEVC, ras.STDDEVI, ras.EXPVALUE
    , ae.RATE                            -- AAL control ONLY; never applied
FROM DBO.RDM_ACCOUNT ra
INNER JOIN DBO.RDM_ACCOUNTSTD ras
       ON  ra.ANLSID      = ras.ANLSID
       AND ra.ID          = ras.ID
       AND ra.PERSPCODE   = ras.PERSPCODE
       AND ra.EVENTID     = ras.EVENTID
       AND ra.PARTITIONID = ras.PARTITIONID    -- guards partition fan-out
INNER JOIN DBO.RDM_ANLSEVENT ae
       ON ae.EVENTID = ra.EVENTID AND ae.ANLSID = ra.ANLSID
WHERE ra.ANLSID = {anlsid} AND ra.PERSPCODE = '{persp}' AND ras.EXPVALUE > 0
  {id_clause}
"""

SQL_MAP = """
SELECT yeqt.SIMID, vem.VENDOREVENTID, yeqt.EVENTID, yeqt.LossQuantile
FROM EVT.VENDOREVENTSMAP vem
INNER JOIN SIM.YEQT yeqt ON yeqt.EVENTID = vem.EVENTID
WHERE vem.HAZARDZONEID = {hz} AND vem.VENDORVERID = {vv} AND yeqt.YEQTID = {yq}
"""

DTYPES = {"SIMID": "int32", "VENDOREVENTID": "int64", "EVENTID": "int64",
          "LossQuantile": "float32"}


def load_elt(anlsid=ANLSID_PORTFOLIO, account_id=None, persp=PERSPCODE):
    """One account (or all, if account_id is None). Always filter before merging."""
    id_clause = f"AND ra.ID = {account_id}" if account_id is not None else ""
    df = pd.read_sql(SQL_ELT.format(anlsid=anlsid, persp=persp, id_clause=id_clause),
                     engine_rdm)
    if account_id is not None and df["VENDOREVENTID"].duplicated().any():
        raise ValueError(f"ELT not unique on event for id={account_id} -- "
                         "check PARTITIONID / PERSPCODE filters")
    return df


def load_mapping(hz=HAZARDZONEID, vv=VENDORVERID, yq=YEQTID):
    """Pulled once and reused. ~3.2M rows; downcast to keep it cheap."""
    df = pd.read_sql(SQL_MAP.format(hz=hz, vv=vv, yq=yq), engine_ref)
    for c, t in DTYPES.items():
        if c in df:
            df[c] = df[c].astype(t)

    dup = df.duplicated(subset=["SIMID", "VENDOREVENTID"]).sum()
    if dup:
        raise ValueError(f"{dup:,} duplicate (SIMID, VENDOREVENTID) rows -- the "
                         "vendor map is not 1:1 at this HAZARDZONEID/VENDORVERID; "
                         "losses would be multiply counted")
    print(f"mapping: {len(df):,} rows | {df['SIMID'].nunique():,} sims | "
          f"{df['VENDOREVENTID'].nunique():,} events | "
          f"{len(df) / df['VENDOREVENTID'].nunique():.1f} sims/event")
    return df

## 3 · Beta fit`sigma = (stdc + stdi) / expvalue` — the standard deviations are **added**, notcombined in quadrature. That is deliberate and follows from the data model: YEQTcarries one `LossQuantile` per (SimID, EventID), shared across every account, sosecondary uncertainty is comonotonic by construction and the independent componentdoes no diversifying work anywhere in the chain. Worth stating explicitly in themethodology doc — it makes portfolio volatility conservative.Where `sigma² ≥ mu(1-mu)` the moment match is infeasible; `clean_params` floorsalpha, which preserves the mean and caps variance at its theoretical maximum. Thatis the right degradation, but it is a real change in shape, so the fraction of AALaffected is reported rather than swallowed.

In [3]:
@dataclass
class BetaFitReport:
    n_rows: int = 0
    n_floored: int = 0
    n_mu_clipped: int = 0
    n_zero_exposure: int = 0
    aal_floored_share: float = 0.0
    def __str__(self):
        return (f"beta fit: {self.n_rows:,} rows | floored {self.n_floored:,} "
                f"({self.aal_floored_share:.2%} of AAL) | mu clipped {self.n_mu_clipped:,} "
                f"| zero exposure {self.n_zero_exposure:,}")


def beta_params(mean, expvalue, stdc, stdi):
    rep = BetaFitReport(n_rows=len(mean))
    mean     = np.asarray(mean, float)
    expvalue = np.asarray(expvalue, float)
    stdc     = np.asarray(stdc, float)
    stdi     = np.asarray(stdi, float)

    bad_exp = expvalue <= 0
    rep.n_zero_exposure = int(bad_exp.sum())
    safe_exp = np.where(bad_exp, 1.0, expvalue)

    mu = mean / safe_exp
    rep.n_mu_clipped = int((mu >= 1.0).sum())
    mu = np.clip(mu, PARAM_FLOOR, MU_CAP)

    sigma = np.maximum((stdc + stdi) / safe_exp, PARAM_FLOOR)

    common = (mu * (1.0 - mu)) / sigma**2 - 1.0
    alpha, beta = mu * common, (1.0 - mu) * common

    infeasible = (alpha <= 0) | (beta <= 0) | ~np.isfinite(alpha) | ~np.isfinite(beta)
    rep.n_floored = int(infeasible.sum())
    if rep.n_floored and mean.sum() > 0:
        rep.aal_floored_share = float(mean[infeasible].sum() / mean.sum())

    a = np.maximum(PARAM_FLOOR, alpha)
    b = np.maximum(PARAM_FLOOR, a * ((1.0 - mu) / mu))
    hit_b = b <= PARAM_FLOOR
    a = np.where(hit_b, np.maximum(PARAM_FLOOR, (mu * b) / (1.0 - mu)), a)

    a = np.where(np.isfinite(a) & (a > 0), a, PARAM_FLOOR)
    b = np.where(np.isfinite(b) & (b > 0), b, PARAM_FLOOR)
    a = np.where(bad_exp, PARAM_FLOOR, a)
    b = np.where(bad_exp, PARAM_FLOOR, b)

    assert np.all(a > 0) and np.all(b > 0), "invalid beta params after cleaning"
    return a, b, rep

## 4 · ELT → YLTOne function, used for **both** the portfolio ELT and each account ELT. Losses area deterministic function of the shared `LossQuantile`, so common random numbers areexact and free — no reseeding, no CRN removal, no bootstrap SEs on the difference.`RATE` is not used here. Frequency comes entirely from the YEQT occurrencestructure; applying the rate again would double-count it.

In [4]:
def simulate_elt(elt: pd.DataFrame, yeqt: pd.DataFrame,
                 sims: np.ndarray | None = None,
                 loss_name: str = "Loss") -> tuple[pd.DataFrame, BetaFitReport]:
    '''
    elt  : VENDOREVENTID, PERSPVALUE, STDDEVC, STDDEVI, EXPVALUE   (unique on event)
    yeqt : VENDOREVENTID, EVENTID, SIMID, LossQuantile
    sims : optional restriction to a set of SimIDs (the window)

    Returns event-level losses keyed on the QBE (SIMID, EVENTID) space.
    '''
    if elt["VENDOREVENTID"].duplicated().any():
        raise ValueError("ELT not unique on VENDOREVENTID -- check ANLSID/ID/"
                         "PERSPCODE/PARTITIONID filters before simulating")

    q = yeqt if sims is None else yeqt[yeqt["SIMID"].isin(sims)]
    j = q.merge(elt, on="VENDOREVENTID", how="inner", validate="m:1")
    if j.empty:
        return pd.DataFrame(columns=["SIMID", "EVENTID", loss_name]), BetaFitReport()

    a, b, rep = beta_params(j["PERSPVALUE"].to_numpy(), j["EXPVALUE"].to_numpy(),
                            j["STDDEVC"].to_numpy(),   j["STDDEVI"].to_numpy())
    j[loss_name] = j["EXPVALUE"].to_numpy() * stats.beta.ppf(
        j["LossQuantile"].to_numpy(), a, b)

    return j[["SIMID", "EVENTID", loss_name]], rep


def reconcile_aal(elt: pd.DataFrame, ylt: pd.DataFrame,
                  n_sims: int = N_SIMS, loss_name: str = "Loss") -> dict:
    '''
    The ONLY use of RATE. Analytic AAL = sum(rate * perspvalue) over the full ELT;
    simulated AAL = total simulated loss / n_sims over the FULL (unwindowed) YLT.
    Divergence implies a mapping error or the wrong VENDORVERID -- this is the
    single strongest control in the pipeline, since frequency is otherwise trusted.
    '''
    analytic  = float((elt["RATE"] * elt["PERSPVALUE"]).sum())
    simulated = float(ylt[loss_name].sum()) / n_sims
    return {"aal_analytic": analytic, "aal_simulated": simulated,
            "rel_diff": abs(simulated - analytic) / analytic if analytic else np.nan}

## 5 · Steps 1–3 · Portfolio YLT and the window

In [5]:
@dataclass
class Window:
    k: int
    rank: int
    n_sims: int
    original_rp_loss: float
    year_oep: pd.Series        # every simulated year
    window_years: np.ndarray   # top-K SimIDs
    events: pd.DataFrame       # SIMID, EVENTID, Loss for those years -- ALL events
    boundary_loss: float       # M_{K+1}
    def __str__(self):
        return (f"window: K={self.k:,} of {len(self.year_oep):,} years | rank "
                f"{self.rank:,} | original {RETURN_PERIOD}RP OEP "
                f"{self.original_rp_loss:,.0f} | boundary {self.boundary_loss:,.0f} "
                f"| {len(self.events):,} (year,event) rows")


def build_window(portfolio_ylt: pd.DataFrame, k: int = K_WINDOW,
                 n_sims: int = N_SIMS, rp: int = RETURN_PERIOD) -> Window:
    year_oep = portfolio_ylt.groupby("SIMID")["Loss"].max()
    rank = int(round(n_sims / rp))

    if k <= rank:
        raise ValueError(f"K={k:,} must exceed target rank {rank:,}")
    if k >= len(year_oep):
        raise ValueError(f"K={k:,} not smaller than simulated years {len(year_oep):,}")

    ordered = year_oep.sort_values(ascending=False)
    window_years = ordered.index[:k].to_numpy()

    events = portfolio_ylt[portfolio_ylt["SIMID"].isin(window_years)][
        ["SIMID", "EVENTID", "Loss"]].copy()

    return Window(k=k, rank=rank, n_sims=n_sims,
                  original_rp_loss=float(ordered.iloc[rank - 1]),
                  year_oep=year_oep, window_years=window_years, events=events,
                  boundary_loss=float(ordered.iloc[k]))

## 6 · Steps 4–8 · Subtract, rerank, certify**A structural point specific to this design.** The portfolio and the account arefitted as *separate* betas and evaluated at the *same* quantile. Nothing guarantees`account_loss(q) ≤ portfolio_loss(q)` row by row — the account's beta can sit abovethe portfolio's at that quantile. This is the same non-linearity your manager noted(`Simulated(P) − Simulated(A) ≠ Simulated(P − A)`), surfacing as negative net losses.Negatives are clipped and **counted**. A small count is numerical; a large count ora large clipped amount means the account is not a subset of that portfolio, or theperspectives differ. Check the diagnostic before trusting any marginal.

In [6]:
@dataclass
class MarginalResult:
    account_id: object
    original_rp_loss: float
    post_removal_rp_loss: float
    marginal: float
    certified: bool
    k: int
    boundary_loss: float
    n_years_touched: int = 0
    n_demotions: int = 0
    n_negative_net: int = 0
    clipped_amount: float = 0.0
    fit_report: BetaFitReport | None = None
    notes: list = field(default_factory=list)
    def __str__(self):
        flag = "exact" if self.certified else "NOT CERTIFIED -- widen K"
        return (f"{self.account_id}: marginal {self.marginal:,.0f} "
                f"({self.original_rp_loss:,.0f} -> {self.post_removal_rp_loss:,.0f}) "
                f"[{flag}] | years touched {self.n_years_touched:,} | "
                f"demotions {self.n_demotions:,} | clipped {self.n_negative_net:,}")


def marginal_impact(window: Window, account_losses: pd.DataFrame,
                    account_id=None) -> MarginalResult:
    ev = window.events.rename(columns={"Loss": "PortfolioLoss"})

    if account_losses.empty:
        m = ev.assign(AccountLoss=0.0)
    else:
        m = ev.merge(account_losses.rename(columns={"Loss": "AccountLoss"}),
                     on=["SIMID", "EVENTID"], how="left", validate="1:1")
        m["AccountLoss"] = m["AccountLoss"].fillna(0.0)

    notes = []
    over = m["AccountLoss"] > m["PortfolioLoss"] + TOL
    n_neg = int(over.sum())
    clipped = float((m.loc[over, "AccountLoss"] - m.loc[over, "PortfolioLoss"]).sum())
    if n_neg:
        notes.append(f"{n_neg:,} rows where account loss exceeded portfolio loss "
                     f"(total {clipped:,.0f}) -- clipped to zero net")
        m.loc[over, "AccountLoss"] = m.loc[over, "PortfolioLoss"]

    m["NetLoss"] = m["PortfolioLoss"] - m["AccountLoss"]

    # per-year max over the FULL event set -- this is what makes it demotion-safe
    agg = m.groupby("SIMID").agg(orig_max=("PortfolioLoss", "max"),
                                 net_max=("NetLoss", "max"),
                                 orig_arg=("PortfolioLoss", "idxmax"),
                                 net_arg=("NetLoss", "idxmax"))

    assert (agg["net_max"] <= agg["orig_max"] + TOL).all(), "removal raised a year loss"

    post_rp = float(np.sort(agg["net_max"].to_numpy())[::-1][window.rank - 1])
    certified = post_rp >= window.boundary_loss - TOL
    if not certified:
        notes.append(f"post-removal RP {post_rp:,.0f} < boundary "
                     f"{window.boundary_loss:,.0f} -- outside years could enter "
                     f"the top {window.rank:,}; widen K")

    return MarginalResult(
        account_id=account_id,
        original_rp_loss=window.original_rp_loss,
        post_removal_rp_loss=post_rp,
        marginal=window.original_rp_loss - post_rp,
        certified=certified, k=window.k, boundary_loss=window.boundary_loss,
        n_years_touched=int((agg["orig_max"] - agg["net_max"] > TOL).sum()),
        n_demotions=int((agg["orig_arg"] != agg["net_arg"]).sum()),
        n_negative_net=n_neg, clipped_amount=clipped, notes=notes)

## 7 · Driver — one account, with auto-widening

In [7]:
def run_account(account_elt: pd.DataFrame, yeqt: pd.DataFrame, window: Window,
                account_id=None, portfolio_ylt: pd.DataFrame | None = None,
                auto_widen: bool = True, max_k: int | None = None) -> MarginalResult:
    '''
    Pass a prebuilt `window` -- steps 1-3 are account-independent and should be
    computed once for the whole run. `portfolio_ylt` is only needed if the
    certificate fails and the window has to be rebuilt wider.
    '''
    win = window
    while True:
        losses, fit = simulate_elt(account_elt, yeqt, sims=win.window_years)
        res = marginal_impact(win, losses, account_id=account_id)
        res.fit_report = fit

        if res.certified or not auto_widen or portfolio_ylt is None:
            return res

        new_k = min(win.k * 2, len(win.year_oep) - 1)
        if max_k is not None:
            new_k = min(new_k, max_k)
        if new_k <= win.k:
            res.notes.append("cannot widen further; marginal is a LOWER BOUND")
            return res
        win = build_window(portfolio_ylt, k=new_k, n_sims=win.n_sims)

## 8 · Production runOrder matters. The portfolio YLT spans **all** years and cannot be windowed —the 200 RP is defined on the full distribution. The window is built once andreused for every account.Memory: filter the ELT to a single `id` *before* merging. Merging the wholeanalysis at once produced 139.8M rows; one account against the mapping is`account_events × ~21 sims/event`.

In [8]:
# --- 0. load once ----------------------------------------------------------
# mapping_df = load_mapping()
#   expect: dup check passes, sims == N_SIMS, sensible sims/event

# --- 1. portfolio ELT -> portfolio YLT (all years) -------------------------
# portfolio_elt = load_elt(account_id=ID_PORTFOLIO)
# portfolio_ylt, pf_fit = simulate_elt(portfolio_elt, mapping_df)
# print(pf_fit)
# print(reconcile_aal(portfolio_elt, portfolio_ylt))   # the mapping control

# --- 2-3. window, once -----------------------------------------------------
# window = build_window(portfolio_ylt)
# print(window)
#
# restrict the mapping to window years ONCE -- every account reuses this
# mapping_win = mapping_df[mapping_df["SIMID"].isin(set(window.window_years))]
# print(f"mapping restricted to window: {len(mapping_win):,} rows "
#       f"({len(mapping_win)/len(mapping_df):.1%} of full)")

# --- 4-8. loop accounts ----------------------------------------------------
# account_ids = sorted(load_elt()["ID"].unique())     # or an explicit list
# results = []
# for acct in account_ids:
#     elt = load_elt(account_id=acct)
#     res = run_account(elt, mapping_win, window, acct, portfolio_ylt=portfolio_ylt)
#     results.append(res)
#     print(res)

# summary = pd.DataFrame([{
#     "account": r.account_id, "marginal": r.marginal, "certified": r.certified,
#     "years_touched": r.n_years_touched, "demotions": r.n_demotions,
#     "clipped_rows": r.n_negative_net, "clipped_amount": r.clipped_amount,
# } for r in results]).sort_values("marginal", ascending=False)
# summary
print("fill in connections above")

fill in connections above


## 9 · Self-test — synthetic universeRuns without server access. Proves the engine is correct by checking the windowedanswer against brute force over **all** simulated years.

In [9]:
rng = np.random.default_rng(20260925)
_N, _E, _RP = 50_000, 400, 200
_RANK = int(round(_N / _RP))

_ev = pd.DataFrame({"VENDOREVENTID": np.arange(1, _E + 1),
                    "EVENTID": np.arange(90_001, 90_001 + _E)})

_occ = [(s, e) for s in range(1, _N + 1)
        for e in rng.choice(_E, size=min(rng.poisson(1.2), 4), replace=False)]
_occ = np.array(_occ)
_yeqt = pd.DataFrame({"SIMID": _occ[:, 0],
                      "VENDOREVENTID": _ev["VENDOREVENTID"].to_numpy()[_occ[:, 1]],
                      "EVENTID":       _ev["EVENTID"].to_numpy()[_occ[:, 1]],
                      "LossQuantile":  rng.uniform(1e-6, 1 - 1e-6, len(_occ))})

def _mk(n, scale, seed):
    r = np.random.default_rng(seed)
    idx  = r.choice(_E, size=n, replace=False)
    exp  = r.uniform(1e5, 5e6, n) * scale
    mean = exp * r.uniform(0.001, 0.25, n)
    return pd.DataFrame({"VENDOREVENTID": _ev["VENDOREVENTID"].to_numpy()[idx],
                         "PERSPVALUE": mean, "EXPVALUE": exp,
                         "STDDEVC": mean * r.uniform(0.2, 1.5, n),
                         "STDDEVI": mean * r.uniform(0.2, 2.0, n),
                         "RATE": r.uniform(1e-6, 5e-4, n)})

_accts = {"A_big": _mk(300, 6.0, 1), "B_mid": _mk(200, 1.0, 2),
          "C_small": _mk(60, 0.2, 3), "D_tiny": _mk(10, 0.02, 4)}

# portfolio ELT = the accounts' events aggregated (stand-in for a real portfolio ELT)
_pf_elt = (pd.concat(_accts.values())
             .groupby("VENDOREVENTID", as_index=False)
             .agg(PERSPVALUE=("PERSPVALUE", "sum"), EXPVALUE=("EXPVALUE", "sum"),
                  STDDEVC=("STDDEVC", "sum"), STDDEVI=("STDDEVI", "sum"),
                  RATE=("RATE", "max")))

_pf_ylt, _pf_fit = simulate_elt(_pf_elt, _yeqt)
print(_pf_fit)

_win = build_window(_pf_ylt, k=5_000, n_sims=_N)
print(_win, "\n")

_res = {}
for _a, _e in _accts.items():
    _r = run_account(_e, _yeqt, _win, _a, portfolio_ylt=_pf_ylt)
    _res[_a] = _r
    print(_r)
    for _n in _r.notes:
        print("    NOTE:", _n)

beta fit: 52,009 rows | floored 12,264 (33.66% of AAL) | mu clipped 0 | zero exposure 0
window: K=5,000 of 32,485 years | rank 250 | original 200RP OEP 28,441,981 | boundary 5,710,108 | 9,788 (year,event) rows 

A_big: marginal 23,517,991 (28,441,981 -> 4,923,990) [exact] | years touched 9,475 | demotions 2,268 | clipped 1,621
    NOTE: 1,621 rows where account loss exceeded portfolio loss (total 469,978,657) -- clipped to zero net
B_mid: marginal 676,877 (28,441,981 -> 27,765,104) [exact] | years touched 2,763 | demotions 28 | clipped 639
    NOTE: 639 rows where account loss exceeded portfolio loss (total 69,517,363) -- clipped to zero net
C_small: marginal 0 (28,441,981 -> 28,441,981) [exact] | years touched 839 | demotions 1 | clipped 135
    NOTE: 135 rows where account loss exceeded portfolio loss (total 1,576,774) -- clipped to zero net
D_tiny: marginal 0 (28,441,981 -> 28,441,981) [exact] | years touched 127 | demotions 1 | clipped 24
    NOTE: 24 rows where account loss exceed

In [10]:
print("--- invariants ---")

assert all(r.marginal >= -TOL for r in _res.values())
print("1. all marginals non-negative                     PASS")

assert all(r.marginal <= r.original_rp_loss + TOL for r in _res.values())
print("2. marginals bounded by original RP loss          PASS")

assert all(r.certified for r in _res.values())
print("3. window sufficiency certified for all accounts  PASS")

# brute force over ALL years -- the decisive check
for _a, _e in _accts.items():
    _full, _ = simulate_elt(_e, _yeqt)
    _m = _pf_ylt.merge(_full.rename(columns={"Loss": "A"}),
                       on=["SIMID", "EVENTID"], how="left")
    _m["A"] = _m["A"].fillna(0.0).clip(upper=_m["Loss"])
    _m["Net"] = _m["Loss"] - _m["A"]
    _brute = float(np.sort(_m.groupby("SIMID")["Net"].max().to_numpy())[::-1][_RANK - 1])
    assert abs(_brute - _res[_a].post_removal_rp_loss) < 1e-6, _a
print("4. window result == brute force over all 50k yrs  PASS")

_tot = sum(r.marginal for r in _res.values())
print(f"5. sum of marginals {_tot:,.0f} vs portfolio 200RP {_win.original_rp_loss:,.0f}"
      f" -> ratio {_tot / _win.original_rp_loss:.3f}")
print("   last-in marginals on a quantile are NOT additive -- expected")

_small = build_window(_pf_ylt, k=_RANK + 5, n_sims=_N)
_rs = run_account(_accts["A_big"], _yeqt, _small, "A_big", auto_widen=False)
print(f"6. undersized window K={_RANK+5}: certified={_rs.certified} "
      f"-> {'correctly rejects' if not _rs.certified else 'DID NOT FIRE'}")

_rw = run_account(_accts["A_big"], _yeqt, _small, "A_big", portfolio_ylt=_pf_ylt)
assert abs(_rw.marginal - _res["A_big"].marginal) < 1e-6
print(f"7. auto-widen recovers exact answer               PASS (final K={_rw.k:,})")

--- invariants ---
1. all marginals non-negative                     PASS
2. marginals bounded by original RP loss          PASS
3. window sufficiency certified for all accounts  PASS


4. window result == brute force over all 50k yrs  PASS
5. sum of marginals 24,194,869 vs portfolio 200RP 28,441,981 -> ratio 0.851
   last-in marginals on a quantile are NOT additive -- expected
6. undersized window K=255: certified=False -> correctly rejects


7. auto-widen recovers exact answer               PASS (final K=8,160)


## 10 · Controls to run before reporting any number| control | where | fails if ||---|---|---|| Partition count = 1 | RMS pre-flight | std join fans out || Map 1:1 on `VENDOREVENTID` | QBE pre-flight | Python merge inflates losses || ELT unique on event | `simulate_elt` | raises || AAL analytic vs simulated | `reconcile_aal` | wrong `VENDORVERID` / mapping gap || `net_max ≤ orig_max` | `marginal_impact` | asserts || Certificate `post_rp ≥ M_{K+1}` | `marginal_impact` | widen K || Clipped-row count ≈ 0 | `marginal_impact` | account not in this portfolio, or perspective mismatch || Both sides at `PERSPCODE='RL'` | manual | marginals won't reconcile to the portfolio |**Open questions for Tim**1. The window: top-K, or the 1000–4000 band in the note? A band excluding ranks   1–999 breaks for exactly the tail-concentrated accounts that matter most.   If 1000–4000 was meant as *smoothing* of a noisy order statistic, that is a   separate and reasonable idea — but it must be applied identically to the   original and post-removal curves, and it still needs a top-K window underneath.2. Expect a high rate of **exact zeros** — any account that never touches a window   year's maximum event scores 0. Report the zero rate as a headline diagnostic.3. Non-additivity: worth pairing this with the existing co-TVaR/Euler allocation   if anyone wants marginals that sum to the portfolio.